# Phase 5 回測與診斷測試

- 主 KPI：`oos_comparison.csv`、`oos_evaluation_summary.md`
- 嚴格 OOS 單例（`evaluate_out_of_sample`）
- `auc_in_sample` = walk-forward CV 均值（非全 train 末段評估）


In [ ]:
import os, sys
from pathlib import Path
import pandas as pd

ROOT = Path(os.getcwd()).resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from backtest.evaluation_utils import BacktestPaths, load_regime_model
from features.feature_utils import FEATURES_PER_SYMBOL
from backtest.single_regime_evaluator import evaluate_on_regime_slice

paths = BacktestPaths.default()
print('backtest root =', paths.root)
print('FEATURES_PER_SYMBOL =', FEATURES_PER_SYMBOL)


In [ ]:
cmp_path = paths.regime_comparison_path()
if cmp_path.is_file():
    df = pd.read_csv(cmp_path)
    display(df.head(20))
    print('rows =', len(df))
else:
    print('請先執行: python backtest_pipeline.py')


In [ ]:
# all_day 模型在 settlement 時段 vs settlement 模型
for label in ['label_realized_volatility']:
    try:
        cross = evaluate_on_regime_slice('BTCUSDT', label, 'all_day', 'settlement', '2024-01-01')
        native = evaluate_on_regime_slice('BTCUSDT', label, 'settlement', 'settlement', '2024-01-01')
        print(label, 'all_day@settlement AUC', cross.get('auc'), 'settlement@settlement', native.get('auc'))
    except Exception as e:
        print('skip', e)


In [ ]:
from pathlib import Path
p = paths.label_equity_path('BTCUSDT', 'settlement', 'label_realized_volatility')
print('equity curve path:', p, 'exists=', p.is_file())


In [ ]:
from backtest.diagnostics import check_time_alignment
from backtest.single_label_evaluator import evaluate_out_of_sample

align = check_time_alignment('BTCUSDT', 'settlement')
print('alignment passed=', align['passed'], 'n_joined=', align.get('n_joined'))

oos = evaluate_out_of_sample(
    'BTCUSDT', 'label_skewness', 'settlement',
    test_start='2025-01-01', train_end='2024-12-31', save_outputs=False,
)
print('OOS status=', oos.get('status'), 'auc_oos=', oos.get('auc_oos'), 'auc_gap=', oos.get('auc_gap'))

In [ ]:
oos_csv = paths.oos_comparison_path()
if oos_csv.is_file():
    oos = pd.read_csv(oos_csv)
    ok = oos[oos['status'] == 'ok']
    print('OOS AUC mean', ok['auc_oos'].mean(), '| CV mean', ok['auc_in_sample'].mean())
    display(ok.groupby(['symbol', 'regime'])['auc_oos'].mean())

summary = paths.oos_evaluation_summary_path()
if summary.is_file():
    print(summary.read_text(encoding='utf-8')[:2000])
else:
    print('請先執行: python backtest_pipeline.py --diagnostic --force')